#### LIBRARY IMPORTS

In [1]:
import pandas as pd
import numpy as np
import glob # For file pattern matching for loading files
import os # Certain debugging operations
import joblib # To save scaler and encoder
import matplotlib.pyplot as plt
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

import copy # for deepcopy in save_results()

# Niave Bayes specifc imports
from sklearn.ensemble import GradientBoostingClassifier

#### CONFIGURATOINS

In [2]:
FEATURE_ROOT = "../speaker_wise-eGeMAPs/functionals/MP4_raw"
OUTPUT_ROOT = "model_outputs/gradient_boosting"
RANDOM_SEED = 42
THRESHOLD = 0.5
GB_N_ESTIMATORS = 100
GB_LEARNING_RATE = 0.1
GB_MAX_DEPTH = 3
GB_SUBSAMPLE = 1.0
GB_MIN_SAMPLES_SPLIT = 2
GB_MIN_SAMPLES_LEAF = 1
TRAIN_RATIO = 0.8
TEST_RATIO = 0.2

CHILD_SPEAKER = {
     
    "speaker_functional_p5-s2.csv": ["SPEAKER_00"],
    "speaker_functional_p5-s7.csv": ["SPEAKER_01"],
    "speaker_functional_p5-s8.csv": ["SPEAKER_05"],
    "speaker_functional_p5-s10.csv": ["SPEAKER_01"],
    "speaker_functional_p5-s13.csv": ["SPEAKER_01"],

    "speaker_functional_p7-s5.csv": ["SPEAKER_01"],
    "speaker_functional_p7-s6.csv": ["SPEAKER_08","SPEAKER_05"],
    # "speaker_functional_p7-s7.csv": ["SPEAKER_02"], # Same as Kiwi
    "speaker_functional_p7-s8.csv": ["SPEAKER_00","SPEAKER_05"],
    "speaker_functional_p7-s16.csv": ["SPEAKER_00"],
    "speaker_functional_p7-s17.csv": ["SPEAKER_00","SPEAKER_02"],
    "speaker_functional_p7-s18.csv": ["SPEAKER_02"],
    "speaker_functional_p7-s29.csv": ["SPEAKER_00"],

    "speaker_functional_p9-s3-1.csv": ["SPEAKER_01"],
    "speaker_functional_p9-s3-2.csv": ["SPEAKER_04"],
    "speaker_functional_p9-s4.csv": ["SPEAKER_06"],
    "speaker_functional_p9-s9.csv": ["SPEAKER_03"],
    "speaker_functional_p9-s15.csv": ["SPEAKER_01"],

    "speaker_functional_p11-s2.csv": ["SPEAKER_02"],
    "speaker_functional_p11-s4.csv": ["SPEAKER_01"],
    "speaker_functional_p11-s8.csv": ["SPEAKER_03"],
    "speaker_functional_p11-s9.csv": ["SPEAKER_00"],
    "speaker_functional_p11-s11.csv": ["SPEAKER_05"],
    "speaker_functional_p11-s15.csv": ["SPEAKER_00"],
    # "speaker_functional_p11-s16-2.csv": ["SPEAKER_01"], # Same as Kiwi
    "speaker_functional_p11-s19.csv": ["SPEAKER_00"],
    "speaker_functional_p11-s22-2.csv": ["SPEAKER_02"],

    "speaker_functional_p12-s2-2.csv": ["SPEAKER_00"],
    # "speaker_functional_p12-s3.csv": ["SPEAKER_01"], # Same as Kiwi
    "speaker_functional_p12-s6.csv": ["SPEAKER_01"],
    # "speaker_functional_p12-s8.csv": ["SPEAKER_00"], # Same as Kiwi
    "speaker_functional_p12-s10.csv": ["SPEAKER_03"],

    "speaker_functional_p17-s2.csv": ["SPEAKER_01"],
    "speaker_functional_p17-s3.csv": ["SPEAKER_04"],
    "speaker_functional_p17-s5.csv": ["SPEAKER_04","SPEAKER_03","SPEAKER_05"],
    "speaker_functional_p17-s6.csv": ["SPEAKER_02"],

    "speaker_functional_p18-s3.csv": ["SPEAKER_00"],
    "speaker_functional_p18-s4.csv": ["SPEAKER_01"],
    # "speaker_functional_p18-s5.csv": ["SPEAKER_00"], # Same as Kiwi
    "speaker_functional_p18-s7.csv": ["SPEAKER_01"],
    "speaker_functional_p18-s8.csv": ["SPEAKER_00"],
    "speaker_functional_p18-s9.csv": ["SPEAKER_01"],
    "speaker_functional_p18-s10.csv": ["SPEAKER_06","SPEAKER_05"],
    "speaker_functional_p18-s11.csv": ["SPEAKER_00"],
    "speaker_functional_p18-s12.csv": ["SPEAKER_00"],
    "speaker_functional_p18-s13.csv": ["SPEAKER_01"],
    "speaker_functional_p18-s15.csv": ["SPEAKER_02","SPEAKER_01"],
    # "speaker_functional_p18-s17.csv": ["SPEAKER_01"], # Same as Kiwi
    "speaker_functional_p18-s18.csv": ["SPEAKER_00"],
    "speaker_functional_p18-s19.csv": ["SPEAKER_02"],
    "speaker_functional_p18-s20.csv": ["SPEAKER_01"],
}


#### FUNCTIONS

In [3]:
def load_participant_data(participant_folder):
    csv_files = [
        f for f in sorted(glob.glob(f"{FEATURE_ROOT}/{participant_folder}/*.csv"))
        if os.path.basename(f) in CHILD_SPEAKER
    ]

    def load_single_speaker(csv_path):
        df = pd.read_csv(csv_path)    
        filename = os.path.basename(csv_path)
        
        target_speakers = CHILD_SPEAKER[filename]
        df = df[df["speaker"].isin(target_speakers)]

        return df
    
    # Train / Validation / Test Split
    
    print(f"{participant_folder}: {len(csv_files)} CSV files")
    
    
    # Load every session for this participant
    full_df = pd.concat(
        [
            load_single_speaker(f)
            for f in csv_files
        ],
        ignore_index=True
    )

    print(f"Total child utterances: {len(full_df)}")
    
    # Remove unnecessary columns
    drop_cols = [
        "participant",
        "session",
        "clip_id",
        "speaker",

        "engagement_start_time",
        "engagement_end_time",

        "speaker_start_time",
        "speaker_end_time",

        "num_segments",
        "speech_duration",
    ]

    full_df = full_df.drop(
        columns=[c for c in drop_cols if c in full_df.columns]
    )

    print(full_df.columns)

    # Separate Features and Labels
    X = full_df.drop(columns=["label"])
    y = full_df["label"]

    return (
        X.reset_index(drop=True),
        y.reset_index(drop=True)
    )

In [4]:
def preprocess_data(X_train, X_test, y_train, y_test):
    
    # Label Encoding
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(y_train)
    y_test = encoder.transform(y_test)

    # Print for debugging purpose
    print(encoder.classes_)
    print(np.unique(y_train))
    print(np.unique(y_test))

    # Feature Scaling
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train) # Learn the Meand and std and transform the data
    X_test = scaler.transform(X_test) # Transform the data using the learned mean and std

    return (
        X_train,
        X_test,
        y_train,
        y_test,
        scaler,
        encoder
    )  

In [5]:
def build_model():

    model = GradientBoostingClassifier(

        n_estimators=GB_N_ESTIMATORS,
        learning_rate=GB_LEARNING_RATE,
        max_depth=GB_MAX_DEPTH,
        subsample=GB_SUBSAMPLE,
        min_samples_split=GB_MIN_SAMPLES_SPLIT,
        min_samples_leaf=GB_MIN_SAMPLES_LEAF,
        random_state=RANDOM_SEED

    )

    return model

In [6]:
def train_model(
        model,
        X_train,
        y_train
):

    model.fit(
        X_train,
        y_train
    )

    return model

In [7]:
def evaluate_model(
        model,
        X_test,
        y_test,
        encoder
):

    # Predict probabilities
    probabilities = model.predict_proba(X_test)[:, 1]

    

    # Convert probabilities to class labels
    predictions = (probabilities >= THRESHOLD).astype(int)

    # True labels
    actual = y_test

    # Confusion Matrix
    cm = confusion_matrix(actual, predictions)

    auroc = roc_auc_score(
        actual,
        probabilities
    )
    
    # Compute metrics
    metrics_dictionary = {
        "Accuracy": accuracy_score(actual, predictions),
        "Precision": precision_score(actual, predictions),
        "Recall": recall_score(actual, predictions),
        "F1 Score": f1_score(actual, predictions),
        "AUROC": auroc
    }

    # Classification report
    report = classification_report(
        actual,
        predictions,
        target_names=encoder.classes_,
        output_dict=True
    )

    print(classification_report(
        actual,
        predictions,
        target_names=encoder.classes_
    ))

    return {
        "metrics": metrics_dictionary,
        "predictions": predictions,
        "probabilities": probabilities,
        "actual": actual,
        "confusion_matrix": cm,
        "classification_report": report
    }

In [8]:
'''
The following are saved
naive_bayes.pkl
scaler.pkl
encoder.pkl
metrics.csv
confusion_matrix.csv
classification_report.csv
predictions.csv
'''
    
def save_results(
        participant,
        model,
        scaler,
        encoder,
        evaluation
):
    
    metrics = evaluation["metrics"]
    predictions = evaluation["predictions"]
    probabilities = evaluation["probabilities"]
    actual = evaluation["actual"]
    cm = evaluation["confusion_matrix"]
    report = evaluation["classification_report"]
    
    # Save Model
    participant_output = Path(OUTPUT_ROOT, participant)
    os.makedirs(participant_output, exist_ok=True)
    joblib.dump(model, Path(participant_output, "gradient-boosting.pkl"))
    
    # Save Scaler
    joblib.dump(scaler,Path(participant_output, "scaler.pkl"))
    
    # Save Encoder
    joblib.dump(encoder,Path(participant_output, "encoder.pkl"))
    
    # Save Metrics
    metrics_df = pd.DataFrame([metrics])

    metrics_df.to_csv(Path(participant_output, "metrics.csv"),index=False)
        
    # Save Confusion Matrix
    cm_df = pd.DataFrame(cm,index=encoder.classes_,columns=encoder.classes_)
    cm_df.to_csv(Path(participant_output, "confusion_matrix.csv"))
    
    # Save classification report
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(Path(participant_output, "classification_report.csv"))
    
    # Save predictions
    prediction_df = pd.DataFrame({
        "Actual": actual,
        "Prediction": predictions,
        "Probability": probabilities
    })

    prediction_df.to_csv(Path(participant_output, "predictions.csv"),index=False)

#### RUN MODEL

In [9]:
summary_results = []

participants = sorted(os.listdir(FEATURE_ROOT))

all_fold_results = []

for participant in participants:

    print(f"Training {participant}")

    # Load ALL data for this participant
    X, y = load_participant_data(participant)

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_SEED
    )

    participant_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

        print(f"\nFold {fold}/5")

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        print("\nTraining class distribution:")
        print(y_train.value_counts())

        print("\nTesting class distribution:")
        print(y_test.value_counts())

        # Preprocess data
        X_train, X_test, y_train, y_test, scaler, encoder = preprocess_data(X_train, X_test, y_train, y_test)

        

        # Build model
        model = build_model()

        # Train model and return training history
        model = train_model(model, X_train, y_train)

        # Evaluate model and return evaluation metrics
        evaluation = evaluate_model(model, X_test, y_test, encoder)
        # Append metrics for this participant to the summary results
        participant_metrics.append(evaluation["metrics"])

        # Append fold results to all_fold_results
        all_fold_results.append({
            "Participant": participant,
            "Fold": fold,
            **evaluation["metrics"]
        })
        
        # Save results
        save_results(
            f"{participant}/fold_{fold}",
            model,
            scaler,
            encoder,
            evaluation
        )
    
    metrics_df = pd.DataFrame(participant_metrics)
    
    summary_results.append({
        "Participant": participant,

        "Accuracy Mean": metrics_df["Accuracy"].mean(),
        "Accuracy Std": metrics_df["Accuracy"].std(),

        "Precision Mean": metrics_df["Precision"].mean(),
        "Precision Std": metrics_df["Precision"].std(),

        "Recall Mean": metrics_df["Recall"].mean(),
        "Recall Std": metrics_df["Recall"].std(),

        "F1 Mean": metrics_df["F1 Score"].mean(),
        "F1 Std": metrics_df["F1 Score"].std(),

        "AUROC Mean": metrics_df["AUROC"].mean(),
        "AUROC Std": metrics_df["AUROC"].std(),
    })
      
pd.DataFrame(all_fold_results).to_csv(
    Path(
        OUTPUT_ROOT,
        "fold_results.csv"
    ),
    index=False
)

# Save summary results as CSV
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv(
    Path(OUTPUT_ROOT,"summary_results.csv"),
    index=False
)
    

Training p11
p11: 8 CSV files
Total child utterances: 381
Index(['F0semitoneFrom27.5Hz_sma3nz_amean',
       'F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
       'F0semitoneFrom27.5Hz_sma3nz_percentile20.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile50.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile80.0',
       'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2',
       'F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope', 'loudness_sma3_amean',
       'loudness_sma3_stddevNorm', 'loudness_sma3_percentile20.0',
       'loudness_sma3_percentile50.0', 'loudness_sma3_percentile80.0',
       'loudness_sma3_pctlrange0-2', 'loudness_sma3_meanRisingSlope',
       'loudness_sma3_stddevRisingSlope', 'loudness_sma3_meanFallingSlope',
       'loudness_sma3_stddevFallingSlope', 'spectralFlux_sma3_amean',
       'spectralFlux_sma3_stddevNorm', '